In [ ]:
# Célula 1: Preparação do Ambiente, Imports e Configurações

import os
from dotenv import load_dotenv
from openai import OpenAI

# 1. Carrega as configurações do arquivo .env
load_dotenv()

# 2. Resgata as variáveis que o professor definiu (visto no seu PDF)
MARITACA_API_KEY = os.getenv("MARITACA_API_KEY")
MARITACA_MODEL_NAME = "sabiazinho-4"
MARITACA_BASE_URL = "https://chat.maritaca.ai/api"

# 3. Validação de segurança que você já possui
if not MARITACA_API_KEY:
    raise ValueError("Não encontrei a variável MARITACA_API_KEY no arquivo .env")

print("Ambiente preparado!")
print(f"Chave carregada com sucesso: {MARITACA_API_KEY[:4]}****")

Ambiente preparado!
Chave carregada com sucesso: 1004****


In [ ]:
# Célula 2: Configuração do Cliente Maritaca

# Inicializa o cliente para conversar com a API da Maritaca
client_maritaca = OpenAI(
    api_key=MARITACA_API_KEY,
    base_url=MARITACA_BASE_URL
)
print("Cliente Maritaca inicializado com sucesso!")

Cliente Maritaca inicializado com sucesso!


In [ ]:
# Célula 3: A Função de Pré-recuperação

def pre_retrieval_expand_query(user_query):
    prompt_sistema = (
        "Você é um assistente especialista em cinema. Seu trabalho é receber uma intenção de busca de filme "
        "e gerar uma lista de termos, sinônimos, diretores ou palavras-chave relacionadas que ajudariam "
        "a encontrar esse filme em um banco de dados. "
        "Responda APENAS com os termos sugeridos, separados por espaço, sem explicações adicionais."
    )
    
    prompt_usuario = f"Expanda os termos de busca para esta intenção: '{user_query}'"
    
    response = client_maritaca.chat.completions.create(
        model=MARITACA_MODEL_NAME,
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": prompt_usuario}
        ],
        temperature=0.3
    )
    
    return response.choices[0].message.content

In [ ]:
# Célula 4: Testando a nossa Pré-recuperação

pergunta_original = "filmes de naves espaciais e robôs"

# Executa a inteligência de Pré-recuperação
pergunta_expandida = pre_retrieval_expand_query(pergunta_original)

print(f"[Original]: {pergunta_original}")
print(f"[Expandida para o Qdrant]: {pergunta_expandida}")

[Original]: filmes de naves espaciais e robôs
[Expandida para o Qdrant]: filmes de ficção científica espaçonaves veículos interestelares robôs androides inteligência artificial aventuras espaciais exploração espacial guerra intergaláctica cyborgs mechas sci-fi space ships androids automação futurista


In [ ]:
# Célula 5: Inicializar o Cliente Qdrant e o Modelo de Embedding

from qdrant_client import QdrantClient

# Endereço padrão e nome da coleção que você já tem no Docker
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "imdb_top_1000_rag"

# Inicializa o cliente do Qdrant
qdrant_client = QdrantClient(url=QDRANT_URL)

# Carrega o modelo de embedding (o mesmo usado na ingestão para os vetores baterem)
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Qdrant e Modelo de Embedding prontos para a busca!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Qdrant e Modelo de Embedding prontos para a busca!


In [ ]:
# Célula 6: Gerar o Vetor e Buscar no Qdrant

# 1. Transforma a pergunta rica em um vetor de 384 dimensões
vetor_pergunta = embedding_model.encode(pergunta_expandida)

# 2. Faz a busca vetorial usando o método correto e atualizado (query_points)
resultados_banco = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=vetor_pergunta,
    limit=10  # Resgatando 10 filmes para o Re-ranqueamento
).points

print(f"Busca realizada! Encontramos {len(resultados_banco)} filmes semelhantes.")
print("\n--- TOP 5 FILMES ENCONTRADOS VIA PRÉ-RECUPERAÇÃO ---")

# Mostra os 5 primeiros usando as chaves corretas do payload
for i, hit in enumerate(resultados_banco[:5]):
    titulo = hit.payload.get('title', 'Sem título')
    diretor = hit.payload.get('director', 'Desconhecido')
    genero = hit.payload.get('genre', 'Não informado')
    print(f"{i+1}. {titulo} | Direção: {diretor} | Gênero: {genero} (Score: {hit.score:.4f})")

Busca realizada! Encontramos 10 filmes semelhantes.

--- TOP 5 FILMES ENCONTRADOS VIA PRÉ-RECUPERAÇÃO ---
1. Kôkaku Kidôtai | Direção: Mamoru Oshii | Gênero: Animation, Action, Crime (Score: 0.4309)
2. Ex Machina | Direção: Alex Garland | Gênero: Drama, Sci-Fi, Thriller (Score: 0.3778)
3. Inception | Direção: Christopher Nolan | Gênero: Action, Adventure, Sci-Fi (Score: 0.3703)
4. Blade Runner | Direção: Ridley Scott | Gênero: Action, Sci-Fi, Thriller (Score: 0.3697)
5. Solaris | Direção: Andrei Tarkovsky | Gênero: Drama, Mystery, Sci-Fi (Score: 0.3679)


In [ ]:
# Célula 7: Carregar o Modelo Cross-Encoder (Re-ranker)

from sentence_transformers import CrossEncoder

print("Carregando o modelo Re-ranker (Cross-Encoder)...")
# Usaremos um modelo clássico e leve da HuggingFace especializado em ranqueamento
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Modelo Re-ranker carregado com sucesso!")

Carregando o modelo Re-ranker (Cross-Encoder)...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\User\Documents\FATESG\Banco de dados não relacional\Código RAG\qdrant\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Modelo Re-ranker carregado com sucesso!


In [ ]:
# Célula 8: Executando o Re-ranqueamento na Prática

# 1. Prepara os pares (Pergunta Original, Texto do Filme) para o Cross-Encoder avaliar
pares_para_avaliacao = []
for hit in resultados_banco:
    # Usamos o 'rag_text' completo do filme para dar todo o contexto ao re-ranker
    texto_filme = hit.payload.get('rag_text', '')
    pares_para_avaliacao.append([pergunta_original, texto_filme])

# 2. O Cross-Encoder calcula as novas notas de relevância
novos_scores = reranker_model.predict(pares_para_avaliacao)

# 3. Associa as novas notas aos respectivos filmes
filmes_reranqueados = []
for i, hit in enumerate(resultados_banco):
    filmes_reranqueados.append({
        "titulo": hit.payload.get('title', 'Sem título'),
        "texto": hit.payload.get('rag_text', ''),
        "score_antigo": hit.score,
        "score_novo": novos_scores[i]
    })

# 4. Ordena a lista do maior score novo para o menor (Re-ranking!)
filmes_reranqueados = sorted(filmes_reranqueados, key=lambda x: x["score_novo"], reverse=True)

print("\n--- TOP 3 FILMES RE-RANQUEADOS (PRONTOS PARA A LLM) ---")
for i, filme in enumerate(filmes_reranqueados[:3]):
    print(f"{i+1}. {filme['titulo']} | Novo Score: {filme['score_novo']:.4f} (Score antigo Qdrant: {filme['score_antigo']:.4f})")


--- TOP 3 FILMES RE-RANQUEADOS (PRONTOS PARA A LLM) ---
1. Solaris | Novo Score: -11.0116 (Score antigo Qdrant: 0.3679)
2. Tropa de Elite | Novo Score: -11.1591 (Score antigo Qdrant: 0.3587)
3. The Iron Giant | Novo Score: -11.1779 (Score antigo Qdrant: 0.3557)


In [ ]:
# Célula 9 (Final): Loop de Atendimento com RAG Avançado

print("=== SISTEMA DE RECOMENDAÇÃO RAG AVANÇADO ATIVO ===")
print("Digite sua pergunta sobre filmes (ou 'sair' para encerrar):\n")

while True:
    # 1. Captura a pergunta do usuário de forma dinâmica
    pergunta_original = input("Sua pergunta: ")
    
    # Condição de parada do loop
    if pergunta_original.lower() == 'sair':
        print("\nSistema encerrado. Até logo!")
        break
        
    if not pergunta_original.strip():
        print("Por favor, digite uma pergunta válida.")
        continue
        
    print("\n[Processando...] Executando etapas do RAG Avançado...")

    try:
        # FASE 1: PRÉ-RECUPERAÇÃO (Query Expansion via Maritaca)
        print("-> Expandindo a consulta...")
        pergunta_expandida = pre_retrieval_expand_query(pergunta_original)
        
        # FASE 2: RECUPERAÇÃO VETORIAL (Qdrant)
        print("-> Buscando candidatos no Qdrant...")
        vetor_pergunta = embedding_model.encode(pergunta_expandida)
        resultados_banco = qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            query=vetor_pergunta,
            limit=10
        ).points
        
        if not resultados_banco:
            print("Nenhum filme encontrado no banco para essa busca.\n")
            continue

        # FASE 3: PÓS-RECUPERAÇÃO (Re-ranqueamento via Cross-Encoder)
        print("-> Re-ranqueando com Cross-Encoder...")
        pares_para_avaliacao = []
        for hit in resultados_banco:
            texto_filme = hit.payload.get('rag_text', '')
            pares_para_avaliacao.append([pergunta_original, texto_filme])
            
        novos_scores = reranker_model.predict(pares_para_avaliacao)
        
        filmes_reranqueados = []
        for i, hit in enumerate(resultados_banco):
            filmes_reranqueados.append({
                "texto": hit.payload.get('rag_text', ''),
                "score_novo": novos_scores[i]
            })
            
        # Ordena do melhor para o pior score do Cross-Encoder
        filmes_reranqueados = sorted(filmes_reranqueados, key=lambda x: x["score_novo"], reverse=True)
        
        # FASE 4: COMPRESSÃO DE CONTEXTO (Pós-recuperação)
        # Filtramos apenas os 3 melhores após a reordenação profunda
        top_3_filmes = filmes_reranqueados[:3]
        contexto_pos_recuperacao = ""
        for i, filme in enumerate(top_3_filmes):
            contexto_pos_recuperacao += f"--- Filme {i+1} ---\n{filme['texto']}\n\n"

        # FASE 5: GERAÇÃO (Maritaca)
        print("-> Gerando resposta final...")
        prompt_sistema_rag = (
            "Você é um sistema de recomendação de filmes inteligente. "
            "Abaixo estão os dados reais do catálogo do IMDb obtidos após um processo de filtragem avançada. "
            "Use APENAS as informações fornecidas no contexto para responder à pergunta do usuário. "
            "Seja direto, amigável e cite os filmes recomendados.\n\n"
            f"CONTEXTO DOS FILMES SELECIONADOS:\n{contexto_pos_recuperacao}"
        )

        resposta_final = client_maritaca.chat.completions.create(
            model=MARITACA_MODEL_NAME,
            messages=[
                {"role": "system", "content": prompt_sistema_rag},
                {"role": "user", "content": pergunta_original}
            ],
            temperature=0.7
        )

        # Exibe o resultado final na tela
        print("\n=== RESPOSTA FINAL DO SISTEMA RAG AVANÇADO ===")
        print(resposta_final.choices[0].message.content)
        print("\n" + "="*50 + "\n")

    except Exception as e:
        print(f"\nOcorreu um erro no processamento: {e}\n")

=== SISTEMA DE RECOMENDAÇÃO RAG AVANÇADO ATIVO ===
Digite sua pergunta sobre filmes (ou 'sair' para encerrar):


[Processando...] Executando etapas do RAG Avançado...
-> Expandindo a consulta...
-> Buscando candidatos no Qdrant...
-> Re-ranqueando com Cross-Encoder...
-> Gerando resposta final...

=== RESPOSTA FINAL DO SISTEMA RAG AVANÇADO ===
Se você gosta de filmes de máfia e gângsters dirigidos por Martin Scorsese, recomendo estes dois clássicos:

1. **Goodfellas (1990)** – Nota 8.7 no IMDb. Conta a história de Henry Hill dentro do mundo do crime italiano-americano, com ótimas atuações de Robert De Niro, Ray Liotta e Joe Pesci.
2. **Casino (1995)** – Nota 8.2 no IMDb. Um thriller sobre poder, ganância e traição no universo dos cassinos controlados pela máfia, estrelado por Robert De Niro, Sharon Stone e Joe Pesci.

Ambos são dirigidos por Scorsese e são referências absolutas do gênero!



[Processando...] Executando etapas do RAG Avançado...
-> Expandindo a consulta...
-> Buscando c